# Lab 2 · Fine-tune (LoRA / QLoRA) + MLflow — Kaggle T4

## What we're doing

We fine-tune **Qwen2.5-3B-Instruct** on the **exact same Kubernetes Q&A data the
Data layer's RAG lab uses** — the public
[`ItshMoh/kubernetes_qa_pairs`](https://huggingface.co/datasets/ItshMoh/kubernetes_qa_pairs)
set indexed in
[`03-data/.../lab1_simple_rag_kaggle.ipynb`](../../../03-data/labs/cloud-kaggle/lab1_simple_rag_kaggle.ipynb).
Then we ask the **same question the RAG lab asks** and compare the two approaches.

**One dataset, two ways to use it:**

- **RAG (03-data · Lab 1)** keeps the answers in a vector store and **retrieves**
  the relevant one *at answer time* — grounded, cited, updatable by re-indexing.
- **Fine-tune (this lab)** bakes them into the **weights** with QLoRA — the model
  answers from memory: no retrieval, no citation, and updating means re-training.

That is the layer's headline rule made tangible: **RAG for knowledge, fine-tune
for behaviour.** Run both on the same question and watch what each gets right.

## What you'll see

- **Cell 2** — load the k8s Q&A set; turn each pair into a chat example.
- **Cell 3 (BEFORE)** — base Qwen2.5-3B on the RAG lab's questions.
- **Cell 4 (TRAIN)** — QLoRA on a single T4 in a few minutes; prints peak VRAM.
- **Cell 5 (AFTER)** — same questions in a fine-tuned k8s voice; compare with the
  RAG lab's grounded + cited answer.
- **Cell 6** — the run logged in MLflow (params + loss), reproducible.

---

**Run from the Kaggle editor on a T4** (Accelerator = *GPU T4 x2*, Internet =
*On*), then **Save Version → Save & Run All**.

## Cell 1 — single GPU + pinned torch + sanity

In [ ]:
import os
# Kaggle 'GPU T4 x2' exposes TWO T4s; Trainer would shard via DataParallel, which
# breaks QLoRA 4-bit ('illegal memory access'). Pin to one GPU. (Set BEFORE torch.)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Pin torch to Kaggle's preinstalled build so pip can't swap in an incompatible wheel.
import torch
open("/tmp/constraints.txt", "w").write("torch==" + torch.__version__.split("+")[0] + "\n")
!pip -q install -c /tmp/constraints.txt trl peft bitsandbytes datasets accelerate mlflow

print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("visible GPUs:", torch.cuda.device_count(), "| torch:", torch.__version__,
      "| capability sm_%d%d" % torch.cuda.get_device_capability())

## Cell 2 — the dataset: the same k8s Q&A the RAG lab uses

We pull [`ItshMoh/kubernetes_qa_pairs`](https://huggingface.co/datasets/ItshMoh/kubernetes_qa_pairs)
— ~500 real Kubernetes question/answer pairs, **the identical source the 03-data
RAG lab indexes.** There, each `answer` is a retrievable chunk; here, each pair
becomes a 3-message chat the model learns to *produce*:

- **system** — `You are a Kubernetes expert…`
- **user** — the `question`
- **assistant** — the gold `answer`

We de-dup on the question and **hold the RAG lab's headline questions out of
training**, so the AFTER check tests generalization, not recitation.

In [ ]:
import json, os, random
from datasets import load_dataset

SYSTEM = "You are a Kubernetes expert. Answer the question clearly and concisely."
# the exact questions the 03-data RAG lab asks — we compare on these at the end
RAG_QUESTIONS = [
    "What is Role-Based Access Control (RBAC) in Kubernetes?",
    "What does the kube-scheduler consider when selecting a node for a Pod?",
]

def to_chat(q, a):
    return {"messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": q},
                         {"role": "assistant", "content": a}]}

raw = load_dataset("ItshMoh/kubernetes_qa_pairs", split="train")
seen, pairs = set(), []
for r in raw:
    q, a = (r.get("question") or "").strip(), (r.get("answer") or "").strip()
    if not q or not a or q.lower() in seen:
        continue
    seen.add(q.lower()); pairs.append((q, a))

# hold the RAG headline questions out of training (honest AFTER check)
held = {q.lower() for q in RAG_QUESTIONS}
heldout = [(q, a) for q, a in pairs if q.lower() in held]   # gold, if present in the set
trainable = [(q, a) for q, a in pairs if q.lower() not in held]
random.seed(0); random.shuffle(trainable)
split = int(len(trainable) * 0.9)
train_pairs, spot_test = trainable[:split], trainable[split:]

os.makedirs("data", exist_ok=True)
with open("data/train.jsonl", "w") as f:
    for q, a in train_pairs: f.write(json.dumps(to_chat(q, a), ensure_ascii=False) + "\n")
with open("data/test.jsonl", "w") as f:
    for q, a in spot_test: f.write(json.dumps(to_chat(q, a), ensure_ascii=False) + "\n")
print(f"train: {len(train_pairs)} | spot-test: {len(spot_test)} | held-out RAG Qs with gold: {len(heldout)}")

print("\n--- one training example (3-message chat) ---")
for m in to_chat(*train_pairs[0])["messages"]:
    print(f"[{m['role']:>9}] {m['content'][:280]}")

## Cell 3 — BEFORE: base Qwen2.5-3B on the RAG lab's questions

The base model already knows some Kubernetes from pre-training, so — unlike the RAG
lab — it won't say *"I don't know."* Watch *how* it answers: verbose, generic, no
citation, no guarantee the specifics are right. That's the gap the fine-tune (a
tighter k8s voice) and RAG (grounding + citations) each close differently.

In [ ]:
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-3B-Instruct"   # same base model as the RAG lab

def gen(tok, model, messages, max_new=220):
    enc = tok.apply_chat_template(messages, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

tok = AutoTokenizer.from_pretrained(MODEL)
base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="auto").eval()
print("== BASE model ==")
for q in RAG_QUESTIONS:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    print("Q:", q)
    print("A:", gen(tok, base, msgs), "\n")
del base; gc.collect(); torch.cuda.empty_cache()

## Cell 4 — Train (adaptive QLoRA/LoRA, fp16) tracked in MLflow

**LoRA** freezes the 3B base and trains tiny low-rank adapters (<1% of params) — so
this finishes in minutes on one GPU. **QLoRA** additionally loads the frozen base in
**4-bit** to fit a T4 (sm_75+); it falls back to plain LoRA elsewhere. We're teaching
the model the *voice and content* of this k8s Q&A set. `report_to="mlflow"` logs
params + the loss curve; watch the loss fall and note the **peak VRAM** at the end.

In [ ]:
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///" + os.path.abspath("mlflow.db")
os.environ["MLFLOW_EXPERIMENT_NAME"] = "qwen-k8s-qa"

from transformers import BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset

OUT = "adapters/lora"
# 4-bit bitsandbytes is reliable on T4-class GPUs (sm_75+); P100 (sm_60) -> plain LoRA fp16.
use_qlora = torch.cuda.get_device_capability()[0] >= 7
method = "qlora" if use_qlora else "lora"
quant = (BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
         if use_qlora else None)
print(f"method: {method}  (base in {'4-bit' if use_qlora else 'fp16'})")

model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=quant,
        dtype=torch.float16, device_map="auto")
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
ds = load_dataset("json", data_files={"train": "data/train.jsonl"})["train"]
# fp16=False on purpose: the trainable LoRA adapters are fp32 and the 4-bit base
# computes in fp16 via bnb, so the fp16 GradScaler isn't needed — and leaving it on
# triggers "_amp_foreach_non_finite_check_and_unscale_ not implemented for BFloat16".
cfg = SFTConfig(output_dir=OUT, num_train_epochs=3,            # QA needs a touch more than format-learning
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        learning_rate=2e-4, warmup_ratio=0.03, logging_steps=5,
        fp16=False, bf16=False, max_length=1024,               # longer than tiny-JSON; no GradScaler
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        report_to="mlflow", run_name=f"{method}-r16", save_strategy="epoch")
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora)
print(f"== training {method} on {len(ds)} examples ==")
trainer.train()
trainer.save_model(OUT); tok.save_pretrained(OUT)
print(f"adapter saved to {OUT} | peak VRAM {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
del trainer, model; gc.collect(); torch.cuda.empty_cache()

## Cell 5 — AFTER: the fine-tuned model on the same questions

Same questions, now through the fine-tuned adapter — the answers should sound more
like this k8s Q&A set (tighter, on-domain). Then compare with how the **RAG lab**
answered the *same* question: retrieved chunks + a **[citation]**, and a clean
*"I don't know"* when the corpus didn't cover it. Fine-tune changed the model's
**behaviour**; RAG supplied the **knowledge**, grounded and cited.

In [ ]:
from peft import PeftModel
ft = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="auto"),
        OUT).eval()
print("== FINE-TUNED ==")
for q in RAG_QUESTIONS:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    print("Q:", q)
    print("A:", gen(tok, ft, msgs), "\n")

# gold answer(s) for any RAG question that was in the dataset (held out of training)
if heldout:
    print("--- gold answer(s), held out of training ---")
    for q, a in heldout:
        print("Q:", q); print("GOLD:", a, "\n")

## Cell 6 — MLflow: what got logged

Kaggle is batch, so there's no live MLflow UI — but the run is fully logged to
`mlflow.db`. Download it from the kernel output and run
`mlflow ui --backend-store-uri sqlite:///mlflow.db` locally to see the loss curve.

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///" + os.path.abspath("mlflow.db"))
exp = mlflow.get_experiment_by_name("qwen-k8s-qa")
runs = mlflow.search_runs([exp.experiment_id])
cols = [c for c in runs.columns if c.startswith(("metrics.", "params."))][:12]
print(runs[["run_id"] + cols].T)

## Takeaways — RAG vs fine-tune, on one dataset

Same `kubernetes_qa_pairs`, two layers, two jobs:

| | Fine-tune (this lab) | RAG (03-data · Lab 1) |
|---|---|---|
| Where knowledge lives | in the **weights** | in a **vector store** |
| At answer time | recall from memory | **retrieve** top-k chunks |
| Citations | none | **[chunk id]** per claim |
| Update a fact | **re-train** | **re-index** (seconds) |
| Out-of-corpus question | answers anyway (may hallucinate) | **"I don't know"** |
| Best at | **behaviour** — tone, format, domain voice | **knowledge** — fresh, private, citable |

Fine-tuning gave the model a Kubernetes *voice*; it did **not** make it a reliable
Kubernetes *source*. For facts that must be correct, current, and cited, reach for
RAG. **RAG for knowledge, fine-tune for behaviour** — now you've run both on the
same data and the same question.